## Introduction
Within the scope of United States history and politics, few social factors have been more powerful than race. Despite this, it is increasingly difficult to measure the effects of race on American society over time throughout the country. People often say that the victor writes history, and in this case, for better or worse, this rings true. Racial discrimination has shaped the history of this country, from city planning to racial gerrymandering; maps have been a tool for imposing discriminatory practices for generations. Today, we will discuss one such mapping practice that has helped enforce racial boundaries across the country, which is now commonly known as Redlining.


In the early 1930s, still reeling from the Great Depression, America faced a housing shortage, and with this struggle, the US government needed to intervene, so in 1934, among the rollout of New Deal policies, the United States Housing Authority, and with it, the Home Owners’ Loan Corporation (HOLC) was established. The HOLC, which operated from 1933 to 1954, dictated how and to whom government mortgages were given. “The government, therefore set out maps of different areas and properties that would be the subject of the above mentioned loans. These maps were color-coded, each color corresponding to the loan worthiness of the neighborhoods in the U.S. and the color red was attributed to the neighborhoods that were deemed not worthy of inclusion in the homeownership programs.” (Cornell Law, 2022) Due to the color coding, this practice gained the name Redlining. These maps systematically segregated many cities across the United States, including my hometown of Chicago, which has been shaped in the image of these maps throughout its history.

## Dataset Overview
In the summer of 2023, students at the University of Richmond worked together to compile many redlining maps from across the country to create the third version of the Mapping Inequality project, helping to visualize these redlining maps and compiling the area descriptions written by the HOLC. My goal was to get a snapshot of how the redlining districts have translated to Chicago’s modern-day neighborhoods, and their economic standings, and so the Mapping Inequality project was where I chose to begin my comparison. To begin, will be using an extracted subset of an existing dataset that was compiled by Mapping Inequality; this subset includes all district entries from the state of Illinois. Due to hardware limitations, I was personally unable to download the entire dataset, so thank you to Prof. Wickes, who helped me extract this data from the existing GitHub data, which is available [here](https://github.com/americanpanorama/mapping-inequality-census-crosswalk/blob/main/MIv3Areas_2020TractCrosswalk.geojson). This larger dataset contains maps from all over the US, so starting with just data from Illinois will also make things easier.


In addition to the Mapping Inequality data, I used data from the Chicago Health Atlas, containing the median income of the census tracts within the city limits. More information will be included later about why I have chosen to use this data but for now, it is available to view [here](https://chicagohealthatlas.org/indicators/INC?tab=map).

## Mapping Inequality Dataset Processing
To begin the process, I started with the Mapping Inequality Dataset. This project was the backbone of my dataset and required the most work, not only to understand but also to process. It's no secret that cities change a lot over time, this posed a major challenge for me when I was trying to adapt the Mapping Inequality dataset. In my initial dataset submission, I attempted to use neighborhoods to divide the city. I learned through that process that I was not able to get enough granular information about the individual districts, and that posed challenges for scaling my dataset so I knew I needed to rethink that approach. In doing so, I began to explore the Chicago Health Atlas data, and when comparing the maps, I began to see some similar shapes. These shapes turned out to be census tracts, and while they do not match up 1:1 with the HOLC's districts, it was a step in the right direction. This realization prompted me to do some further digging in the Mapping Inequalities codebase. In doing so, I found their census crosswalk JSON data. This file was the basis for the rest of this dataset.

In [1]:
import json

with open("illinois.geojson", "rt", encoding="utf-8") as map:
    data = json.load(map)

For those unfamiliar with JSON or Javascript Orientation Notation is a lightweight data interchange format designed to be extremely flexible and adaptable for different types of data and widely usable across different programming languages. This flexibility is a great strength of the JSON format, but it means every JSON file can look distinctly different. The "illinois.geojson" data that we have here includes metadata about the districts defined by the HOLC in Chicago. An individual district entry looks like this:


{
           "type": "Feature",
           "properties": {
               "area_id": 1581,
               "city": "Chicago",
               "state": "IL",
               "city_survey": true,
               "cat": "Hazardous",
               "grade": "D",
               "label": "D99",
               "res": true,
               "com": false,
               "ind": false,
               "fill": "#d9838d",
               "GISJOIN": "G1700310460200",
               "GEOID": "17031460200",
               "calc_area": 2484.33122,
               "pct_tract": 0.00414
           },
           "geometry": {
               "type": "MultiPolygon",
               "coordinates": [
                   [
                       [
                           [
                               -87.55194,
                               41.74462
                           ],
                           [
                               -87.551917,
                               41.74406
                           ],
                           [
                               -87.552126,
                               41.744177
                           ],
                           [
                               -87.552618,
                               41.744455
                           ],
                           [
                               -87.552899,
                               41.744614
                           ],
                           [
                               -87.55194,
                               41.74462
                           ]
                       ]
                   ]
               ]
           }
       },


Each entry contains 3 key pieces of information, firstly a distinction of what it is (in this case "Feature") and two types of dictionaries, one named "properties", which contains detailed information about the District, and information key to determining it's grade, the other, named "geometry," contains geographic placement information used to display the districts; however, it was not useful for my purposes, so it is safe to ignore.

In [2]:
import pandas as pd

map_df = pd.json_normalize(data)
map_df

,type,name,features,crs.type,crs.properties.name
0,FeatureCollection,MIv3Areas_2020TractCrosswalk_pres6,"[{'type': 'Feature', 'properties': {'area_id':...",name,urn:ogc:def:crs:OGC:1.3:CRS84


In [3]:
features_df = pd.json_normalize(map_df['features']).T.reset_index()
features_df

,index,0
0,0,"{'type': 'Feature', 'properties.area_id': 1052..."
1,1,"{'type': 'Feature', 'properties.area_id': 1043..."
2,2,"{'type': 'Feature', 'properties.area_id': 1043..."
3,3,"{'type': 'Feature', 'properties.area_id': 1043..."
4,4,"{'type': 'Feature', 'properties.area_id': 1055..."
...,...,...
4389,4389,"{'type': 'Feature', 'properties.area_id': 1155..."
4390,4390,"{'type': 'Feature', 'properties.area_id': 1171..."
4391,4391,"{'type': 'Feature', 'properties.area_id': 1381..."
4392,4392,"{'type': 'Feature', 'properties.area_id': 1170..."


Once I understood the structure of the JSON data thoroughly enough, my goal was to extract all the information from the 'properties' category and convert it into a format I could later merge with the Chicago Health Atlas data. I used Pandas to reshape the data into a dataframe, making it easier to understand how the information was being stored. Additionally, I was able to collect only entries from Chicago, further narrowing our dataset.

In [4]:
properties_df = pd.json_normalize(data['features'])
properties_df = properties_df[[col for col in properties_df.columns if col.startswith('properties.')]]
properties_df


,properties.area_id,properties.city,properties.state,properties.city_survey,properties.cat,properties.grade,properties.label,properties.res,properties.com,properties.ind,properties.fill,properties.GISJOIN,properties.GEOID,properties.calc_area,properties.pct_tract
0,1052,Aurora,IL,True,Best,A,A1,True,False,False,#76a865,G1700890852905,17089852905,146955.98872,0.04047
1,1043,Aurora,IL,True,Best,A,A2,True,False,False,#76a865,G1700890853006,17089853006,4406.39412,0.00284
2,1043,Aurora,IL,True,Best,A,A2,True,False,False,#76a865,G1700890853100,17089853100,11551.29962,0.00804
3,1043,Aurora,IL,True,Best,A,A2,True,False,False,#76a865,G1700890853900,17089853900,973472.11757,0.33439
4,1055,Aurora,IL,True,Still Desirable,B,B,True,False,False,#7cb5bd,G1700890853005,17089853005,466.41558,0.00019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4389,1155,Chicago,IL,True,Still Desirable,B,B43,True,False,False,#7cb5bd,None,None,12699.43430,NaN
4390,1171,Chicago,IL,True,Still Desirable,B,B71,True,False,False,#7cb5bd,None,None,14215.56519,NaN
4391,1381,Chicago,IL,True,Still Desirable,B,B78,True,False,False,#7cb5bd,None,None,33384.77302,NaN
4392,1170,Chicago,IL,True,Definitely Declining,C,C64,True,False,False,#ffff00,None,None,339.67124,NaN


In [5]:
chicago_df = properties_df[properties_df['properties.city'].str.contains('Chicago', na='False')]
chicago_df

,properties.area_id,properties.city,properties.state,properties.city_survey,properties.cat,properties.grade,properties.label,properties.res,properties.com,properties.ind,properties.fill,properties.GISJOIN,properties.GEOID,properties.calc_area,properties.pct_tract
104,11474,Chicago,IL,True,Definitely Declining,C,315,True,False,False,#ffff00,G1700310818900,17031818900,18709.61512,0.01262
105,11474,Chicago,IL,True,Definitely Declining,C,315,True,False,False,#ffff00,G1700310819000,17031819000,689159.27522,0.10979
106,11474,Chicago,IL,True,Definitely Declining,C,315,True,False,False,#ffff00,G1700310819500,17031819500,1965.61097,0.00134
107,11474,Chicago,IL,True,Definitely Declining,C,315,True,False,False,#ffff00,G1700310819700,17031819700,4863.08945,0.00200
108,1065,Chicago,IL,True,Best,A,A1,True,False,False,#76a865,G1700970863400,17097863400,858564.65613,0.16619
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4389,1155,Chicago,IL,True,Still Desirable,B,B43,True,False,False,#7cb5bd,None,None,12699.43430,NaN
4390,1171,Chicago,IL,True,Still Desirable,B,B71,True,False,False,#7cb5bd,None,None,14215.56519,NaN
4391,1381,Chicago,IL,True,Still Desirable,B,B78,True,False,False,#7cb5bd,None,None,33384.77302,NaN
4392,1170,Chicago,IL,True,Definitely Declining,C,C64,True,False,False,#ffff00,None,None,339.67124,NaN


Once I got to this point, I also began to think about how I would choose to merge things. Specifically, I was looking for any piece of information our two datasets had in common.

## Reading In The City of Chicago Data

Now that we have our map data in a format we can use, we need to read in our CSV data from the Chicago Health Atlas. The Chicago Health Atlas is an effort maintained by the Chicago Department of Public Health and UIC's PHAME center, aiming to provide accurate data about Chicago's communities. They collect data on several metrics, including


In [6]:
import csv
with open("chicago_census_tracts.csv",'rt', encoding='utf-8') as infile:
    headers, *data = csv.reader(infile)

tracts_df = pd.DataFrame(data, columns=headers)
tracts_df

,﻿Layer,Name,GEOID,INC_2020-2024,INC_2020-2024_moe
0,Census tract,Tract 101,17031010100,"$74,145","$3,495"
1,Census tract,Tract 102.01,17031010201,"$63,056","$21,160"
2,Census tract,Tract 102.02,17031010202,"$53,220","$14,868"
3,Census tract,Tract 103,17031010300,"$73,105","$16,064"
4,Census tract,Tract 104,17031010400,"$54,083","$9,232"
...,...,...,...,...,...
782,Census tract,Tract 8439,17031843900,"$56,923","$18,229"
783,Census tract,Tract 8446,17031844600,"$58,144","$17,292"
784,Census tract,Tract 8447,17031844700,"$60,250","$32,177"
785,Census tract,Tract 8400,17043840000,"$90,795","$24,446"


Again, for those unfamiliar with the format, a Comma Separated Values file or CSV is a file format designed to store tabular data where each field is separated by a comma, hence the name. In this case, there isn't much reformatting to do, which is wonderful! This data is in a form that is already usable, so I didn't have to go through the hassle of reshaping again, as I did with the Mapping Inequality data.

## Merging the Datasets
Now that both of our datasets are in usable formats, it is up to us to merge them and decide what columns will be important for analysis later.

In [7]:
chicago_df = chicago_df.rename(columns={'properties.GEOID': 'GEOID'})
chicago_df = chicago_df[chicago_df['GEOID'] != 'NONE']

In [8]:
chicago_df.shape

(3525, 15)

After some more time staring at this dataset and figuring out what was going on, I decided I needed to make a few final changes to both tables, these changes included removing all empty GEOID values in the chicago_df table to reduce bloat in the final dataset. I also renamed the properties.GEOID column in the chicago_df to just GEOID to remain consistent with the column name in the tracts_df.

In [9]:
merged_df = pd.merge(tracts_df, chicago_df, how='inner', on='GEOID')
merged_df = merged_df.reset_index(drop=True)
merged_df = merged_df.loc[merged_df['properties.grade'].notna()]


I also chose to remove all null property grades or labels. Though it only looks like we removed grades, this is because all entries without one were also missing the other. I chose to remove them since it would be impossible to know where these districts are with no reference.

In [10]:
merged_df

,﻿Layer,Name,GEOID,INC_2020-2024,INC_2020-2024_moe,properties.area_id,properties.city,properties.state,properties.city_survey,properties.cat,properties.grade,properties.label,properties.res,properties.com,properties.ind,properties.fill,properties.GISJOIN,properties.calc_area,properties.pct_tract
0,Census tract,Tract 101,17031010100,"$74,145","$3,495",1171,Chicago,IL,True,Still Desirable,B,B71,True,False,False,#7cb5bd,G1700310010100,37552.88911,0.09895
1,Census tract,Tract 101,17031010100,"$74,145","$3,495",1170,Chicago,IL,True,Definitely Declining,C,C64,True,False,False,#ffff00,G1700310010100,209064.67634,0.55088
2,Census tract,Tract 101,17031010100,"$74,145","$3,495",1166,Chicago,IL,True,Definitely Declining,C,C65,True,False,False,#ffff00,G1700310010100,1726.09768,0.00455
3,Census tract,Tract 101,17031010100,"$74,145","$3,495",1172,Chicago,IL,True,Definitely Declining,C,C69,True,False,False,#ffff00,G1700310010100,26852.27807,0.07076
4,Census tract,Tract 102.01,17031010201,"$63,056","$21,160",1148,Chicago,IL,True,Still Desirable,B,B44,True,False,False,#7cb5bd,G1700310010201,7466.30282,0.01480
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2293,Census tract,Tract 8439,17031843900,"$56,923","$18,229",1507,Chicago,IL,True,Definitely Declining,C,C220,True,False,False,#ffff00,G1700310843900,307623.91872,0.18924
2294,Census tract,Tract 8446,17031844600,"$58,144","$17,292",1492,Chicago,IL,True,Hazardous,D,D73,True,False,False,#d9838d,G1700310844600,183274.31517,0.42164
2295,Census tract,Tract 8446,17031844600,"$58,144","$17,292",1488,Chicago,IL,True,Hazardous,D,D74,True,False,False,#d9838d,G1700310844600,251190.51195,0.57789
2297,Census tract,Tract 8447,17031844700,"$60,250","$32,177",1329,Chicago,IL,True,Definitely Declining,C,C162,True,False,False,#ffff00,G1700310844700,5752.85448,0.01432


In [11]:
merged_df = merged_df.drop(columns=['Layer', 'properties.city', 'properties.state', 'properties.city_survey'], errors='ignore')
merged_df

,﻿Layer,Name,GEOID,INC_2020-2024,INC_2020-2024_moe,properties.area_id,properties.cat,properties.grade,properties.label,properties.res,properties.com,properties.ind,properties.fill,properties.GISJOIN,properties.calc_area,properties.pct_tract
0,Census tract,Tract 101,17031010100,"$74,145","$3,495",1171,Still Desirable,B,B71,True,False,False,#7cb5bd,G1700310010100,37552.88911,0.09895
1,Census tract,Tract 101,17031010100,"$74,145","$3,495",1170,Definitely Declining,C,C64,True,False,False,#ffff00,G1700310010100,209064.67634,0.55088
2,Census tract,Tract 101,17031010100,"$74,145","$3,495",1166,Definitely Declining,C,C65,True,False,False,#ffff00,G1700310010100,1726.09768,0.00455
3,Census tract,Tract 101,17031010100,"$74,145","$3,495",1172,Definitely Declining,C,C69,True,False,False,#ffff00,G1700310010100,26852.27807,0.07076
4,Census tract,Tract 102.01,17031010201,"$63,056","$21,160",1148,Still Desirable,B,B44,True,False,False,#7cb5bd,G1700310010201,7466.30282,0.01480
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2293,Census tract,Tract 8439,17031843900,"$56,923","$18,229",1507,Definitely Declining,C,C220,True,False,False,#ffff00,G1700310843900,307623.91872,0.18924
2294,Census tract,Tract 8446,17031844600,"$58,144","$17,292",1492,Hazardous,D,D73,True,False,False,#d9838d,G1700310844600,183274.31517,0.42164
2295,Census tract,Tract 8446,17031844600,"$58,144","$17,292",1488,Hazardous,D,D74,True,False,False,#d9838d,G1700310844600,251190.51195,0.57789
2297,Census tract,Tract 8447,17031844700,"$60,250","$32,177",1329,Definitely Declining,C,C162,True,False,False,#ffff00,G1700310844700,5752.85448,0.01432


After review, I decided to remove the following columns because they all contain the same value for each entry:


   - properties.city
   - properties.state
   - properties.city_survey


I ran into issues removing the "Layer" column for a reason I was unable to identify, though I initially planned to drop this column as well for the same reason.

In [12]:
merged_df.to_csv('final-data-set.csv')

Our final step is to review that we have the proper dataset and make a standalone file out of our merged dataframe. More information about the dataset, and the specifics of the statistics it holds, is located [here](https://github.com/CultureAsData-UIUC/is310-spring-2026-group-6/tree/main/final_datasets/Adin)

## Discussion & Conclusions
This dataset is not perfect, and as I sit and reflect there are lots of things I'd like to add or change, after all this is my first time trying to do something like this. For this final discussion section, I'd like to discuss some of what I've learned, what I believe I could have done better, and what I would do if given more time to perfect this dataset. I hope I get to work with similar data in the future, it helped that I have some personal connection and was able to bring in some prior knowledge, but still this project presented challenges I wasn't prepared for and I think they're important to cover for anyone who might be trying to do something similar in the future.

#### What I Learned
Curating a dataset was an entirely new experience for me, the process not only challenged me to think deeply about what data to use, but also challenged me to remain intentional throughout the process and have a strong rational for why I chose what I did. When I was asked to make a dataset about history my immediate thought was to try and datafy something race related and over spring break I began to really think on it. This is also when I was exposed to the Mapping Inequality project giving me headway on a thought. My family has been Chicago natives for generations now, on both sides of my family, I currently reside on the South side and through my childhood I've gotten the chance to experience living in several of Chicago's many diverse neighborhoods. Each neighborhood has a distinct identity shaping the culture and feeling of each one, I believe this is a beautiful thing about the city and makes it really unique. Desipte this, I am also faced with the reality that Chicago's history has shaped it to be an incredibly segregated city. Through my studies in highschool I was exposed to the practice of redlining and it provided a lot of answers for my own lived experience.The other benefit of starting this project over spring break was that I got the chance to talk to some of my family members about what I was doing. They all had their opinions about what I was working on and I'd like to continue this project eventually to provide more context. 

#### Limitations
In my discussions with my family they they also brought up a very valid point about a limitation of my data: using only snapshots of times in Chicago's history does not tell the full story of how these areas of the city have changed over time. One limitation of the current dataset is that it does not indicate which neighborhoods each census tract is in, making it a bit more difficult to interpret the data without background information. I also believe that this data could benefit from some form of visualization to represent how much change has actually occurred between time periods. I would like to continue to update this data as new stuff is released, and eventually, maybe try to recover some more historical data to show a more holistic picture of how Chicago has changed over time. All of these things are entirely outside the scope of this project (and frankly, my technical ability), but I think they're important to note nonetheless.

#### Notable Neighborhoods
Ultimately, in some ways, this project has left me with more questions than answers. Through my time looking at all this data, I have begun to notice patterns and interesting, albeit anecdotal, outliers. For example, district D104 known both then and now as Mount Greenwood, is an interesting neighborhood.

In [13]:
merged_df[merged_df['properties.label'] == 'D104']

,﻿Layer,Name,GEOID,INC_2020-2024,INC_2020-2024_moe,properties.area_id,properties.cat,properties.grade,properties.label,properties.res,properties.com,properties.ind,properties.fill,properties.GISJOIN,properties.calc_area,properties.pct_tract
1883,Census tract,Tract 7401,17031740100,"$107,500","$20,942",1599,Hazardous,D,D104,True,False,False,#d9838d,G1700310740100,8.787816e+05,0.98592
1885,Census tract,Tract 7402,17031740200,"$129,207","$9,520",1599,Hazardous,D,D104,True,False,False,#d9838d,G1700310740200,1.184502e+06,0.60345
1886,Census tract,Tract 7403,17031740300,"$118,717","$19,251",1599,Hazardous,D,D104,True,False,False,#d9838d,G1700310740300,1.063146e+06,0.80497
1887,Census tract,Tract 7404,17031740400,"$89,250","$29,610",1599,Hazardous,D,D104,True,False,False,#d9838d,G1700310740400,8.924224e+05,0.31437
1942,Census tract,Tract 8233.04,17031823304,"$51,662","$10,028",1599,Hazardous,D,D104,True,False,False,#d9838d,G1700310823304,6.128434e+04,0.01725


This neighborhood is currently one of the most affluent neighborhoods in the city, on average, despite it's D rating. Other Neighborhoods like Lincoln Park are drastically split in where they lie, both in grade,a dn current median income.

In [14]:
lincoln_park = ['A30','D11','D12','D13','D14','D15']
merged_df[merged_df['properties.label'].isin(lincoln_park)]

,﻿Layer,Name,GEOID,INC_2020-2024,INC_2020-2024_moe,properties.area_id,properties.cat,properties.grade,properties.label,properties.res,properties.com,properties.ind,properties.fill,properties.GISJOIN,properties.calc_area,properties.pct_tract
153,Census tract,Tract 314,17031031400,"$93,026","$12,239",1368,Best,A,A30,True,False,False,#76a865,G1700310031400,14738.29733,0.00990
253,Census tract,Tract 608,17031060800,"$90,174","$12,341",1368,Best,A,A30,True,False,False,#76a865,G1700310060800,32974.74085,0.18169
256,Census tract,Tract 609,17031060900,"$79,016","$8,512",1368,Best,A,A30,True,False,False,#76a865,G1700310060900,70669.75077,0.14160
266,Census tract,Tract 619.01,17031061901,"$73,769","$14,115",1368,Best,A,A30,True,False,False,#76a865,G1700310061901,7004.14980,0.04369
271,Census tract,Tract 619.02,17031061902,"$95,705","$18,011",1368,Best,A,A30,True,False,False,#76a865,G1700310061902,61924.29960,0.22403
306,Census tract,Tract 632,17031063200,"$79,782","$13,135",1368,Best,A,A30,True,False,False,#76a865,G1700310063200,29.49326,0.00008
322,Census tract,Tract 702,17031070200,"$109,000","$28,125",1395,Hazardous,D,D12,True,False,False,#d9838d,G1700310070200,59029.82543,0.17290
323,Census tract,Tract 702,17031070200,"$109,000","$28,125",1397,Hazardous,D,D15,True,False,False,#d9838d,G1700310070200,417.64543,0.00122
325,Census tract,Tract 703,17031070300,"$156,898","$67,260",1395,Hazardous,D,D12,True,False,False,#d9838d,G1700310070300,154248.89376,0.47548
328,Census tract,Tract 704,17031070400,"$192,955","$80,243",1395,Hazardous,D,D12,True,False,False,#d9838d,G1700310070400,154707.15252,0.47696


Again, despite several placements with a D grade, many of the districts are incredibly high income, while some of the A districts are much lower income by comparison. Many area descriptions included notes about "Shifting or Infiltration." This field noted whether new racial/ethnic groups were moving into the district, either through internal shifts or through immigration. A trend I noted while compiling my initial dataset was that not only were there almost no A-graded districts within City limits, with only 8 total, but they also rarely had any notes about shifting or infiltration; those that did noted that they were of "Desirable Class." D graded districts, on the other hand, almost all had notes about shifting or infiltration. In the case of Mount Greenwood, it received a D grade because of Lithuanian and Irish immigrants. This highlights a key distinction between today's perception of race and ethnicity and that of the 1930s and 40s. I can tell you from experience that many people in this neighborhood identify as Irish, and despite this, they are now categorized by the census as white, despite other racial groups trending in the opposite direction, opting for more granular options about ethnic identity.

#### Conclusion
So, here's the part where I have to try to neatly tie this up with a bow, even though I have so many more questions now than when I started this project. While I'd like to say I have some major revelation from this data, I truly believe there is more analysis necessary before I would be able to do so. I would also like to address the fact that this final dataset is quite different from the initial dataset that I collected. Unfortunately, I was unable to find a computational solution for including the elements of the Area Description for each district in my final dataset. As I reflect on the project now, I would've loved to find a solution so I could include that information, as I believe it would have provided additional insights about this data, but unfortunately, I'm out of time. Ultimately, I believe there is an incredible amount of room for additional scholarly work in this space. I hope I am able to continue contributing to that body of work as I continue my career. I believe that better understanding the effects of things like redlining can be a key piece in helping understand the larger effects of race on American history.